In this exercise, we will predict the number of applications received
using the other variables in the College data set.

(a) Split the data set into a training set and a test set.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

url = "https://www.statlearning.com/s/College.csv"
college = pd.read_csv(url, index_col=0)

y = college["Apps"]
X = college.drop(columns=["Apps"])

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)


(b) Fit a linear model using least squares on the training set, and
report the test error obtained.

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred_lm = lm.predict(X_test)
mse_lm = mean_squared_error(y_test, y_pred_lm)
print("Linear regression test MSE:", mse_lm)


Linear regression test MSE: 642753.8976533666


(c) Fit a ridge regression model on the training set, with λ chosen
by cross-validation. Report the test error obtained.

In [3]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

alphas = np.logspace(10, -5, 1000)

ridge = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=alphas, scoring="neg_mean_squared_error", cv=10))
])

ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
best_alpha = ridge.named_steps["ridge"].alpha_
print("Ridge best lambda (alpha):", best_alpha)
print("Ridge test MSE:", mse_ridge)


Ridge best lambda (alpha): 1e-05
Ridge test MSE: 642753.9091042259


(d) Fit a lasso model on the training set, with λ chosen by crossvalidation. Report the test error obtained, along with the number of non-zero coefficient estimates.

In [4]:
from sklearn.linear_model import LassoCV

lasso = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(alphas=alphas, cv=10, max_iter=100000,
                      random_state=1))
])

lasso.fit(X_train, y_train)

y_pred_lasso = lasso.predict(X_test)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
best_alpha_lasso = lasso.named_steps["lasso"].alpha_
coef_lasso = lasso.named_steps["lasso"].coef_

n_nonzero = (coef_lasso != 0).sum()

print("Lasso best lambda (alpha):", best_alpha_lasso)
print("Lasso test MSE:", mse_lasso)
print("Number of non-zero coefficients:", n_nonzero)


Lasso best lambda (alpha): 12.476595526308685
Lasso test MSE: 660054.043834101
Number of non-zero coefficients: 14


(e) Fit a PCR model on the training set, with M chosen by crossvalidation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [5]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

n_features = X_train.shape[1]

pcr = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("lm", LinearRegression())
])

param_grid = {
    "pca__n_components": list(range(1, n_features + 1))
}

pcr_cv = GridSearchCV(
    pcr,
    param_grid,
    scoring="neg_mean_squared_error",
    cv=10
)

pcr_cv.fit(X_train, y_train)

best_M = pcr_cv.best_params_["pca__n_components"]
print("Best number of components M:", best_M)

y_pred_pcr = pcr_cv.predict(X_test)
mse_pcr = mean_squared_error(y_test, y_pred_pcr)
print("PCR test MSE:", mse_pcr)


Best number of components M: 17
PCR test MSE: 642753.8976533791
